# Train & Export Upgraded Stockfish-Distilled Value CNN (`nojs/chess` · v2)

This notebook builds a **192,000-board** chess evaluation & blunder-contrast dataset labeled by **Stockfish**, trains a higher-capacity **BatchNorm-free 3-Layer Quantized Convolutional Value Network (`SVGChessValueCNN`, `2,623` parameters)** with **8-bit Straight-Through Quantization-Aware Training (`ste_q8`)** + **Counterfactual Material/QxN Monotonicity**, verifies parity against the exact HTML5 SVG `<filter>` 2-ply Minimax engine, and exports `weights.json` for `chess/`.

### Key Upgrades for 1100–1300+ Elo Play & Anti-Blunder Discipline
1. **Nearly 2× Model Capacity (`1,345 → 2,623` Params) + `5×5` Knight-Vision Receptive Field**:
   - **Layer 1 (`8×8 → 4×4`)**: `SVGPointwise1x1(6 → 18)` (`6` RGB tiles) + **`5×5` depthwise `<feConvolveMatrix>` (`padding=2`)** + `2×2` AvgPool. Upgrading Layer 1 from `3×3` to `5×5` lets the first spatial layer directly see **all 8 Knight jumps `(±1, ±2), (±2, ±1)`**, castled King pawn shields, pins, forks, and 2-square sliding threats at exact 1-square resolution before pooling (with zero cross-board bleed thanks to our `10×10` 2-pixel zero moat!).
   - **Layer 2 (`4×4 → 2×2`)**: `SVGPointwise1x1(18 → 27)` (`9` RGB tiles) + `3×3` depthwise `<feConvolveMatrix>` (`padding=1`) + `2×2` AvgPool.
   - **Layer 3 (`2×2 → 1×1`)**: `SVGPointwise1x1(27 → 36)` (`12` RGB tiles) + `2×2` global depthwise `<feConvolveMatrix>`.
   - **Lossless Linear Material & Queen-Conservation Skip (`6 → 3`)**: Dedicated linear channels for classical material (`P=1, N=3.1, B=3.3, R=5, Q=9.4`) and **Queen/Major-vs-Minor conservation** so losing a Queen for a Knight always carries an unmistakable global linear signature.
   - **Calibrated Value Head (`39 → 1`)**: `13` RGB `<feColorMatrix>` triplets summed around `Z0 = 128/255` + 256-entry Sigmoid LUT (`[3..253]/255`, sharper `310 cp` WDL scale).
2. **2× Larger Dataset (`192,000` Boards) with Stockfish Play, Intentional Bad Positional Moves, Unsound `QxN` Trades & Random Blunders**:
   - **Realistic Stockfish & ECO Opening Rollouts**: Classical human openings (Italian, Ruy Lopez, Sicilian, French, Caro-Kann, Queen's Gambit, London, Scandinavian, Four Knights, etc.) played out by persistent Stockfish UCI workers.
   - **Intentional Bad Positional Moves (`MultiPV` bottom quiet moves)**: Stockfish intentionally plays anti-positional moves (weakening King pawn shields `f6/g5`, rim Knights `Na3/Nh6`, premature Queen exposure, self-blocking bishops/pawns) so the value network learns to penalize bad positional structure.
   - **Explicit Unsound Trades (`QxN → PxQ`, `QxB`, `RxN`, Hanging Pieces) & Random Blunders**: Directly constructs **2-ply Minimax contrastive leaf pairs `(s → m1_good → m2_best)` vs `(s → m1_bad → m2_punish)`** from the same parent board `s`, teaching the network never to sacrifice a Queen for a Knight or hang material without real Stockfish-verified compensation.
   - **Counterfactual Piece-Drop & `QxN` Monotonicity Loss**: Enforces strict Ceteris-Paribus logit margins whenever an AI piece is removed or an AI Queen is traded for an Opponent Knight on the exact same board.

**Usage in Google Colab**: Select a **T4 GPU** runtime and click **Runtime $\rightarrow$ Run all** (~5–8 minutes end-to-end) to install Stockfish, generate & label the 192k corpus, train with 8-bit QAT, verify anti-blunder tactics, and automatically download `weights.json`.


In [ ]:
# ---------------------------------------------------------------------------
# 0. Install Stockfish Binary & python-chess on Google Colab
# ---------------------------------------------------------------------------
import os
import platform
import shutil
import stat
import subprocess
import sys
import tarfile
import urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "chess"])

def ensure_stockfish() -> str:
    """Locate or install a fast Stockfish UCI binary on Colab / Linux / macOS."""
    for candidate in (shutil.which("stockfish"), "/usr/games/stockfish", "/usr/local/bin/stockfish", "./stockfish_bin"):
        if candidate and os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate

    # 1. Fast Ubuntu/Debian apt install on Google Colab (~4s)
    if shutil.which("apt-get"):
        print("Installing Stockfish via apt-get...")
        res = subprocess.run(
            ["apt-get", "update", "-qq"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        if res.returncode == 0:
            subprocess.run(
                ["apt-get", "install", "-y", "-qq", "stockfish"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
        for candidate in (shutil.which("stockfish"), "/usr/games/stockfish"):
            if candidate and os.path.isfile(candidate):
                return candidate

    # 2. Direct official GitHub release binary fallback for Linux x86_64
    if platform.system() == "Linux" and platform.machine() in ("x86_64", "AMD64"):
        print("Downloading official Stockfish Linux x86-64 AVX2 release...")
        url = "https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar"
        tar_path = "stockfish_avx2.tar"
        urllib.request.urlretrieve(url, tar_path)
        with tarfile.open(tar_path, "r") as tf:
            for member in tf.getmembers():
                if member.isfile() and "stockfish" in os.path.basename(member.name) and not member.name.endswith(".tar"):
                    extracted = tf.extractfile(member)
                    if extracted is not None:
                        with open("./stockfish_bin", "wb") as out_f:
                            out_f.write(extracted.read())
                        os.chmod("./stockfish_bin", os.stat("./stockfish_bin").st_mode | stat.S_IEXEC)
                        return "./stockfish_bin"

    raise RuntimeError("Could not find or install Stockfish binary!")

STOCKFISH_PATH = ensure_stockfish()
print(f"Stockfish ready at: {STOCKFISH_PATH}")


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
import json
import math
import random
import time
from typing import Dict, List, Optional, Sequence, Tuple

import chess
import chess.engine
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Exact 8-bit integer zero-point: 128 / 255 avoids the 127.5 half-integer rounding drift!
ZERO_POINT_U8 = 128
ZERO_POINT = round(ZERO_POINT_U8 / 255.0, 6)  # 0.501961
POS_VAL_U8 = 255                              # +127/255 above Z0 (AI piece)
NEG_VAL_U8 = 1                                # -127/255 below Z0 (Opponent piece)
CP_WDL_SCALE = 310.0                          # Sharper 1-pawn (+21 LSB) & 1-minor (+58 LSB) WDL resolution!

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {device} | CPU count: {os.cpu_count()} | Z0 = {ZERO_POINT} ({ZERO_POINT_U8}/255)")


In [ ]:
# ---------------------------------------------------------------------------
# 1. Stockfish-Guided Position Generator with Intentional Bad Positional Moves,
#    Unsound Material Trades (QxN -> PxQ), & Random Blunders (~192,000 Boards)
# ---------------------------------------------------------------------------

PIECE_TYPES = (chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING)
PIECE_VAL = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}

# 28 Classical 1100-1800 Elo Opening Book Stems (ensures realistic human structures!)
OPENING_BOOK_UCI: List[List[str]] = [
    [],
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "f8c5"],          # Giuoco Piano / Italian
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1c4", "g8f6"],          # Two Knights Defense
    ["e2e4", "e7e5", "g1f3", "b8c6", "f1b5", "a7a6", "b5a4"],  # Ruy Lopez
    ["e2e4", "e7e5", "g1f3", "b8c6", "d2d4", "e5d4", "f3d4"],  # Scotch Game
    ["e2e4", "e7e5", "g1f3", "b8c6", "b1c3", "g8f6"],          # Four Knights
    ["e2e4", "e7e5", "f1c4", "b8c6", "d1h5", "g7g6", "h5f3"],  # Wayward Queen / Scholar's Defense
    ["e2e4", "e7e5", "g1f3", "d8h4", "b1c3"],                  # Premature Black Queen sortie
    ["e2e4", "c7c5", "g1f3", "d7d6", "d2d4", "c5d4", "f3d4"],  # Open Sicilian
    ["e2e4", "c7c5", "g1f3", "b8c6", "f1b5"],                  # Rossolimo Sicilian
    ["e2e4", "c7c5", "c2c3", "g8f6", "e4e5", "f6d5"],          # Alapin Sicilian
    ["e2e4", "e7e6", "d2d4", "d7d5", "b1c3", "g8f6"],          # Classical French
    ["e2e4", "e7e6", "d2d4", "d7d5", "e4e5", "c7c5"],          # Advance French
    ["e2e4", "c7c6", "d2d4", "d7d5", "b1c3", "d5e4", "c3e4"],  # Classical Caro-Kann
    ["e2e4", "d7d5", "e4d5", "d8d5", "b1c3", "d5a5"],          # Scandinavian (Queen active!)
    ["e2e4", "d7d6", "d2d4", "g8f6", "b1c3", "g7g6"],          # Pirc Defense
    ["d2d4", "d7d5", "c2c4", "e7e6", "b1c3", "g8f6"],          # Queen's Gambit Declined
    ["d2d4", "d7d5", "c2c4", "c7c6", "g1f3", "g8f6"],          # Slav Defense
    ["d2d4", "d7d5", "c2c4", "d5c4", "g1f3", "g8f6"],          # Queen's Gambit Accepted
    ["d2d4", "d7d5", "f1f4", "g8f6", "e2e3", "c7c5"],          # London System
    ["d2d4", "g8f6", "c2c4", "g7g6", "b1c3", "f8g7", "e2e4"],  # King's Indian Defense
    ["d2d4", "g8f6", "c2c4", "e7e6", "b1c3", "f8b4"],          # Nimzo-Indian Defense
    ["d2d4", "g8f6", "c2c4", "g7g6", "b1c3", "d7d5"],          # Grunfeld Defense
    ["c2c4", "e7e5", "b1c3", "g8f6", "g2g3", "d7d5"],          # Reversed Sicilian / English
    ["g1f3", "d7d5", "g2g3", "g8f6", "f1g2", "c7c6"],          # King's Indian Attack / Reti
    ["e2e4", "e7e5", "f2f4", "e5f4", "g1f3"],                  # King's Gambit
    ["d2d4", "f7f5", "c2c4", "g8f6", "g2g3"],                  # Dutch Defense
    ["e2e4", "b8c6", "d2d4", "d7d5"],                          # Nimzowitsch Defense
]


def encode_board_uint8(board: chess.Board, ai_color: Optional[chess.Color] = None) -> np.ndarray:
    """Encode `board` into uint8 `(6, 8, 8)` from `ai_color`'s perspective (defaults to `board.turn`)."""
    persp = board.turn if ai_color is None else ai_color
    arr = np.full((6, 8, 8), ZERO_POINT_U8, dtype=np.uint8)
    for sq, pc in board.piece_map().items():
        r, f = divmod(sq, 8)
        ar = r if persp == chess.WHITE else (7 - r)
        arr[pc.piece_type - 1, ar, f] = POS_VAL_U8 if pc.color == persp else NEG_VAL_U8
    return arr


def cp_to_wdl(cp: float, scale: float = CP_WDL_SCALE) -> float:
    """Map Stockfish centipawns (side-to-move POV) to smooth WDL win probability in [0.005, 0.995]."""
    x = max(-18.0, min(18.0, cp / scale))
    w = 1.0 / (1.0 + math.exp(-x))
    return float(min(0.995, max(0.005, w)))


def _find_unsound_sacrifices_and_hangs(board: chess.Board) -> List[Tuple[chess.Move, int]]:
    """Find legal moves that make bad trades (especially QxN, QxB, QxP, RxN, RxB on defended squares)
    or hang a Queen/Rook/Minor on an attacked square with zero material gain.
    Returns list of `(bad_move, net_material_loss_estimate)` sorted worst-first.
    """
    opp = not board.turn
    cands: List[Tuple[chess.Move, int]] = []
    for mv in board.legal_moves:
        p_att = board.piece_type_at(mv.from_square)
        if p_att is None or p_att == chess.KING:
            continue
        v_att = PIECE_VAL[p_att]
        p_vic = board.piece_type_at(mv.to_square)
        v_vic = PIECE_VAL.get(p_vic, 0) if p_vic is not None else 0
        if mv.promotion is not None:
            continue
        # Test if moving `mv` leaves our piece capturable on `mv.to_square`!
        b1 = board.copy(stack=False)
        b1.push(mv)
        if b1.is_checkmate():
            continue
        if b1.is_attacked_by(opp, mv.to_square):
            net_loss = v_att - v_vic
            if net_loss >= 2:
                # Boost priority of Queen-for-Knight / Queen-for-Bishop / Queen-hang blunders!
                priority = net_loss + (4 if p_att == chess.QUEEN else 0)
                cands.append((mv, priority))
    cands.sort(key=lambda t: t[1], reverse=True)
    return cands


def _best_recapture_or_punish(b1: chess.Board, target_sq: int, eng: chess.engine.SimpleEngine, lim_fast: chess.engine.Limit) -> Optional[chess.Move]:
    """Pick the cleanest legal recapture on `target_sq` (LVA first), or Stockfish's best punishing reply."""
    leg = list(b1.legal_moves)
    if not leg:
        return None
    recaps = [m for m in leg if m.to_square == target_sq and m.promotion in (None, chess.QUEEN)]
    if recaps:
        recaps.sort(key=lambda m: PIECE_VAL.get(b1.piece_type_at(m.from_square) or chess.QUEEN, 9))
        return recaps[0]
    res = eng.play(b1, lim_fast)
    return res.move if res.move is not None else leg[0]


def _synthesize_endgame_or_imbalance_board(rng: random.Random) -> Optional[chess.Board]:
    """Construct valid endgame & post-trade material imbalance boards (including Q vs N, Q vs B, Q vs R!)."""
    templates = [
        ([chess.QUEEN], []),
        ([chess.ROOK], []),
        ([chess.QUEEN, chess.PAWN, chess.PAWN], [chess.KNIGHT, chess.PAWN, chess.PAWN]),       # Q vs N imbalance!
        ([chess.QUEEN, chess.KNIGHT, chess.PAWN], [chess.BISHOP, chess.KNIGHT, chess.PAWN]),   # Q vs B imbalance!
        ([chess.QUEEN, chess.PAWN], [chess.ROOK, chess.PAWN]),                                 # Q vs R imbalance!
        ([chess.QUEEN, chess.PAWN], [chess.ROOK, chess.KNIGHT]),                               # Q+P vs R+N
        ([chess.ROOK, chess.PAWN, chess.PAWN], [chess.KNIGHT, chess.PAWN, chess.PAWN]),        # R vs N exchange!
        ([chess.ROOK, chess.PAWN], [chess.BISHOP, chess.PAWN]),
        ([chess.BISHOP, chess.KNIGHT, chess.PAWN], [chess.ROOK]),
        ([chess.PAWN, chess.PAWN], [chess.PAWN]),
        ([chess.KNIGHT, chess.PAWN, chess.PAWN], [chess.BISHOP, chess.PAWN]),
        ([], []),
    ]
    strong_pcs, weak_pcs = rng.choice(templates)
    w_pcs, b_pcs = (strong_pcs, weak_pcs) if rng.random() < 0.5 else (weak_pcs, strong_pcs)

    for _ in range(30):
        b = chess.Board.empty()
        sqs = rng.sample(range(64), 2 + len(w_pcs) + len(b_pcs))
        wk, bk = sqs[0], sqs[1]
        if chess.square_distance(wk, bk) <= 1:
            continue
        b.set_piece_at(wk, chess.Piece(chess.KING, chess.WHITE))
        b.set_piece_at(bk, chess.Piece(chess.KING, chess.BLACK))
        ok, idx = True, 2
        for col, plist in ((chess.WHITE, w_pcs), (chess.BLACK, b_pcs)):
            for pt in plist:
                sq = sqs[idx]; idx += 1
                if pt == chess.PAWN and chess.square_rank(sq) in (0, 7):
                    ok = False; break
                b.set_piece_at(sq, chess.Piece(pt, col))
            if not ok:
                break
        if not ok:
            continue
        b.turn = chess.WHITE if rng.random() < 0.5 else chess.BLACK
        if b.is_valid():
            return b
    return None


def generate_stockfish_and_blunder_corpus(
    sf_path: str,
    target_singletons: int = 120000,
    target_pairs: int = 36000,
    seed: int = SEED,
) -> Tuple[List[str], List[Tuple[str, str]], List[str]]:
    """Parallel Stockfish-guided game & blunder generator producing:
      - `single_fens` (`120,000`): realistic Stockfish play + intentional bad positional moves +
        post-QxN/blunder recaptures + random-move perturbations + endgames
      - `contrast_pairs` (`36,000` pairs = `72,000` boards):
        * 2-ply `(s -> m1_good -> m2_best, s -> m1_unsound_trade_or_hang -> m2_recap)` (AI turn at leaf!)
        * 2-ply `(s -> m1_good_quiet -> m2_best, s -> m1_bad_positional_or_random -> m2_best)` (AI turn at leaf!)
        * 1-ply sibling `(s -> m_a, s -> m_b)`
    """
    num_workers = max(2, min(6, (os.cpu_count() or 2)))
    rng_main = random.Random(seed)
    seen_keys = set()
    single_fens: List[str] = []
    regimes: List[str] = []
    contrast_pairs: List[Tuple[str, str]] = []

    def _try_add(fen: str, key: str, reg: str) -> None:
        if len(single_fens) < target_singletons and key not in seen_keys:
            seen_keys.add(key)
            single_fens.append(fen)
            regimes.append(reg)

    # 1. Synthesize clean endgames & post-trade imbalances (~16% of singletons)
    n_end_target = int(0.16 * target_singletons)
    while len(single_fens) < n_end_target:
        eb = _synthesize_endgame_or_imbalance_board(rng_main)
        if eb is not None:
            k = eb.board_fen() + (" w" if eb.turn == chess.WHITE else " b")
            _try_add(eb.fen(), k, "endgame")

    rem_sing_per_worker = math.ceil((target_singletons - len(single_fens)) / num_workers) + 1500
    pairs_per_worker = math.ceil(target_pairs / num_workers) + 400

    def _rollout_worker(w_id: int) -> Tuple[List[Tuple[str, str, str]], List[Tuple[str, str]]]:
        w_rng = random.Random(seed + 1009 * (w_id + 1))
        w_singles: List[Tuple[str, str, str]] = []
        w_pairs: List[Tuple[str, str]] = []
        eng = chess.engine.SimpleEngine.popen_uci(sf_path)
        try:
            eng.configure({"Threads": 1, "Hash": 16})
            lim_fast = chess.engine.Limit(nodes=450)
            lim_mpv = chess.engine.Limit(nodes=650)

            while len(w_singles) < rem_sing_per_worker or len(w_pairs) < pairs_per_worker:
                b = chess.Board()
                # Start 75% of games from a classical human opening stem (with small prefix truncation)
                if w_rng.random() < 0.75:
                    stem = w_rng.choice(OPENING_BOOK_UCI)
                    cut = w_rng.randint(0, len(stem)) if stem else 0
                    for uci in stem[:cut]:
                        mv_b = chess.Move.from_uci(uci)
                        if mv_b in b.legal_moves:
                            b.push(mv_b)

                max_plies = w_rng.randint(24, 92)
                for ply in range(max_plies):
                    if b.is_game_over():
                        k = b.board_fen() + (" w" if b.turn == chess.WHITE else " b")
                        w_singles.append((b.fen(), k, "terminal"))
                        break

                    leg = list(b.legal_moves)
                    if not leg:
                        break

                    # --- A. Harvest 2-Ply Unsound-Trade / Hanging-Piece Contrastive Leaf Pairs! ---
                    unsound = _find_unsound_sacrifices_and_hangs(b)
                    if unsound and len(w_pairs) < pairs_per_worker and w_rng.random() < 0.42:
                        m1_good_res = eng.play(b, lim_fast)
                        m1_good = m1_good_res.move if m1_good_res.move is not None else leg[0]
                        m1_bad = unsound[0][0] if w_rng.random() < 0.75 else w_rng.choice(unsound)[0]
                        if m1_good != m1_bad:
                            # Good 2-ply leaf: s -> m1_good -> Stockfish best reply
                            bg = b.copy(stack=False); bg.push(m1_good)
                            if not bg.is_game_over():
                                r_good = eng.play(bg, lim_fast).move or next(iter(bg.legal_moves))
                                bg.push(r_good)
                                # Bad 2-ply leaf: s -> m1_bad (e.g. QxN!) -> Opponent recapture (PxQ!)
                                bb = b.copy(stack=False); bb.push(m1_bad)
                                r_bad = _best_recapture_or_punish(bb, m1_bad.to_square, eng, lim_fast)
                                if r_bad is not None:
                                    bb.push(r_bad)
                                    w_pairs.append((bg.fen(), bb.fen()))
                                    kb = bb.board_fen() + (" w" if bb.turn == chess.WHITE else " b")
                                    w_singles.append((bb.fen(), kb, "bad_trade"))

                    # --- B. Harvest 2-Ply Positional Good-vs-Bad (Stockfish MultiPV best vs worst quiet!) ---
                    elif len(leg) >= 5 and len(w_pairs) < pairs_per_worker and w_rng.random() < 0.22:
                        infos = eng.analyse(b, lim_mpv, multipv=min(6, len(leg)))
                        pvs = [inf["pv"][0] for inf in infos if "pv" in inf and inf["pv"]]
                        if len(pvs) >= 2:
                            m1_best = pvs[0]
                            # Intentional bad positional move (Stockfish bottom MultiPV choice or random quiet move!)
                            quiets = [m for m in leg if not b.is_capture(m) and m != m1_best]
                            m1_bad_pos = pvs[-1] if (w_rng.random() < 0.65 or not quiets) else w_rng.choice(quiets)
                            if m1_best != m1_bad_pos:
                                bg = b.copy(stack=False); bg.push(m1_best)
                                bb = b.copy(stack=False); bb.push(m1_bad_pos)
                                if not bg.is_game_over() and not bb.is_game_over():
                                    rg = eng.play(bg, lim_fast).move or next(iter(bg.legal_moves))
                                    rb = eng.play(bb, lim_fast).move or next(iter(bb.legal_moves))
                                    bg.push(rg); bb.push(rb)
                                    w_pairs.append((bg.fen(), bb.fen()))
                                    kp = bb.board_fen() + (" w" if bb.turn == chess.WHITE else " b")
                                    w_singles.append((bb.fen(), kp, "bad_pos"))

                    # --- C. Choose Game Rollout Policy (Mix Strong Stockfish, Intentional Bad Moves, & Random!) ---
                    roll = w_rng.random()
                    used_tag = "middlegame"
                    if roll < 0.58:
                        # Realistic Stockfish move (mix top-1 and MultiPV-2/3 human-like variety)
                        if w_rng.random() < 0.70 or len(leg) < 3:
                            mv = eng.play(b, lim_fast).move or w_rng.choice(leg)
                        else:
                            infos = eng.analyse(b, lim_fast, multipv=min(3, len(leg)))
                            pvs = [inf["pv"][0] for inf in infos if "pv" in inf and inf["pv"]]
                            mv = w_rng.choice(pvs) if pvs else w_rng.choice(leg)
                        used_tag = "opening" if ply < 14 else ("endgame" if len(b.piece_map()) <= 10 else "sf_play")
                    elif roll < 0.76 and len(leg) >= 4:
                        # Stockfish intentionally plays a weak/anti-positional move!
                        infos = eng.analyse(b, lim_fast, multipv=min(6, len(leg)))
                        pvs = [inf["pv"][0] for inf in infos if "pv" in inf and inf["pv"]]
                        mv = pvs[-1] if len(pvs) >= 2 else w_rng.choice(leg)
                        used_tag = "bad_pos"
                    elif roll < 0.86 and unsound:
                        # Execute an unsound trade / piece hang and let Stockfish punish it on next ply!
                        mv = unsound[0][0]
                        used_tag = "bad_trade"
                    else:
                        # Pure random legal move injected mid-game!
                        mv = w_rng.choice(leg)
                        used_tag = "random_mv"

                    b.push(mv)
                    if w_rng.random() < 0.48 or used_tag in ("bad_pos", "bad_trade", "random_mv"):
                        k = b.board_fen() + (" w" if b.turn == chess.WHITE else " b")
                        w_singles.append((b.fen(), k, used_tag))
        finally:
            eng.quit()
        return w_singles, w_pairs

    t0 = time.time()
    print(f"Generating Stockfish + intentional bad-move / unsound-trade corpus across {num_workers} UCI workers...")
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        results = list(pool.map(_rollout_worker, range(num_workers)))

    for w_singles, w_pairs in results:
        for fen, k, reg in w_singles:
            _try_add(fen, k, reg)
        for pair in w_pairs:
            if len(contrast_pairs) < target_pairs:
                contrast_pairs.append(pair)

    rng_main.shuffle(contrast_pairs)
    contrast_pairs = contrast_pairs[:target_pairs]
    print(
        f"Generated {len(single_fens):,} unique positions + {len(contrast_pairs):,} contrastive 2-ply pairs "
        f"({len(single_fens) + 2 * len(contrast_pairs):,} total boards) in {time.time() - t0:.1f}s"
    )
    for reg in sorted(set(regimes)):
        cnt = regimes.count(reg)
        print(f"  Regime {reg:11s}: {cnt:6,d} ({100.0 * cnt / len(regimes):4.1f}%)")
    return single_fens, contrast_pairs, regimes


SINGLE_FENS, SIBLING_PAIRS, REGIMES = generate_stockfish_and_blunder_corpus(
    STOCKFISH_PATH,
    target_singletons=120000,
    target_pairs=36000,
)


In [ ]:
# ---------------------------------------------------------------------------
# 2. Deduplicated Parallel Persistent-UCI Stockfish Distillation Labeling
# ---------------------------------------------------------------------------

SF_NODES_LIMIT = 2200  # Deeper ~2,200 NNUE nodes/pos for crisp tactical & positional ground truth!

def label_fens_with_stockfish(fens: Sequence[str], sf_path: str, nodes_limit: int = SF_NODES_LIMIT) -> Tuple[np.ndarray, np.ndarray]:
    """Deduplicate `fens` by `(board_fen, turn)` so Stockfish evaluates each unique board once,
    then scatter `(encoded_boards_u8, target_wdl_f32)` back to all `len(fens)` slots.
    """
    n_total = len(fens)
    unique_fens: List[str] = []
    key_to_uidx: Dict[str, int] = {}
    slot_to_uidx = np.empty(n_total, dtype=np.int32)

    for i, fen in enumerate(fens):
        parts = fen.split(" ")
        key = parts[0] + " " + parts[1] + " " + parts[2] + " " + parts[3]
        uidx = key_to_uidx.get(key)
        if uidx is None:
            uidx = len(unique_fens)
            key_to_uidx[key] = uidx
            unique_fens.append(fen)
        slot_to_uidx[i] = uidx

    n_uniq = len(unique_fens)
    ux_u8 = np.empty((n_uniq, 6, 8, 8), dtype=np.uint8)
    uy_wdl = np.empty(n_uniq, dtype=np.float32)

    num_workers = max(2, min(6, (os.cpu_count() or 2)))
    chunk_size = math.ceil(n_uniq / num_workers)

    def _worker(w_id: int, start: int, end: int) -> None:
        eng = chess.engine.SimpleEngine.popen_uci(sf_path)
        try:
            eng.configure({"Threads": 1, "Hash": 32})
            lim = chess.engine.Limit(nodes=nodes_limit)
            for idx in range(start, end):
                bd = chess.Board(unique_fens[idx])
                ux_u8[idx] = encode_board_uint8(bd, ai_color=bd.turn)
                if bd.is_checkmate():
                    uy_wdl[idx] = 0.005
                elif bd.is_stalemate() or bd.is_insufficient_material():
                    uy_wdl[idx] = 0.500
                else:
                    info = eng.analyse(bd, lim)
                    pov_score = info["score"].pov(bd.turn)
                    mate_n = pov_score.mate()
                    if mate_n is not None:
                        uy_wdl[idx] = 0.995 if mate_n > 0 else 0.005
                    else:
                        cp = float(pov_score.score() or 0.0)
                        uy_wdl[idx] = cp_to_wdl(cp)
                if w_id == 0 and ((idx - start + 1) % 5000 == 0):
                    pct = 100.0 * (idx - start + 1) / max(1, end - start)
                    print(f"  [Worker 0] Labeled {idx - start + 1:,}/{end - start:,} ({pct:.0f}%)...")
        finally:
            eng.quit()

    t0 = time.time()
    print(f"Labeling {n_uniq:,} unique boards ({n_total:,} total instances) with Stockfish ({nodes_limit} nodes/pos)...")
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futs = []
        for w_id in range(num_workers):
            s_i = w_id * chunk_size
            e_i = min(n_uniq, (w_id + 1) * chunk_size)
            if s_i < e_i:
                futs.append(pool.submit(_worker, w_id, s_i, e_i))
        for fut in futs:
            fut.result()

    elapsed = time.time() - t0
    print(f"Stockfish labeled {n_uniq:,} unique boards in {elapsed:.1f}s ({n_uniq / max(elapsed, 1e-3):,.0f} pos/sec across {num_workers} UCI workers)")
    return ux_u8[slot_to_uidx], uy_wdl[slot_to_uidx]


all_pair_fens = [f for pair in SIBLING_PAIRS for f in pair]
combined_fens = SINGLE_FENS + all_pair_fens
x_all_u8, y_all_wdl = label_fens_with_stockfish(combined_fens, STOCKFISH_PATH, nodes_limit=SF_NODES_LIMIT)

n_sing = len(SINGLE_FENS)
x_sing_u8, y_sing_wdl = x_all_u8[:n_sing], y_all_wdl[:n_sing]
x_pairs_u8 = x_all_u8[n_sing:].reshape(len(SIBLING_PAIRS), 2, 6, 8, 8)
y_pairs_wdl = y_all_wdl[n_sing:].reshape(len(SIBLING_PAIRS), 2)

# Stratified 84% Train / 8% Val / 8% Test split on singletons + 90/10 split on 2-ply contrastive pairs
rng_np = np.random.default_rng(SEED)
perm = rng_np.permutation(n_sing)
n_train = int(0.84 * n_sing)
n_val = int(0.08 * n_sing)
tr_idx, va_idx, te_idx = perm[:n_train], perm[n_train : n_train + n_val], perm[n_train + n_val :]

x_train_f = x_sing_u8[tr_idx].astype(np.float32) / 255.0
y_train_f = y_sing_wdl[tr_idx]
x_val_f = x_sing_u8[va_idx].astype(np.float32) / 255.0
y_val_f = y_sing_wdl[va_idx]
x_test_f = x_sing_u8[te_idx].astype(np.float32) / 255.0
y_test_f = y_sing_wdl[te_idx]
regimes_test = [REGIMES[i] for i in te_idx]

n_pair_tr = int(0.90 * len(SIBLING_PAIRS))
xp_train_f = x_pairs_u8[:n_pair_tr].astype(np.float32) / 255.0
yp_train_f = y_pairs_wdl[:n_pair_tr]
xp_val_f = x_pairs_u8[n_pair_tr:].astype(np.float32) / 255.0
yp_val_f = y_pairs_wdl[n_pair_tr:]

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train_f), torch.from_numpy(y_train_f)), batch_size=BATCH_SIZE, shuffle=True)
pair_loader = DataLoader(TensorDataset(torch.from_numpy(xp_train_f), torch.from_numpy(yp_train_f)), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val_f), torch.from_numpy(y_val_f)), batch_size=512, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.from_numpy(x_test_f), torch.from_numpy(y_test_f)), batch_size=512, shuffle=False)

plt.figure(figsize=(7.8, 2.8))
plt.hist(y_train_f, bins=50, color="#38bdf8", edgecolor="#0f172a", alpha=0.85)
plt.axvline(0.5, color="#fbbf24", linestyle="--", label="Equal (0.0 cp)")
plt.title(f"Stockfish Distilled Target WDL Distribution ({len(y_train_f):,} Train Singletons + {n_pair_tr:,} 2-Ply Pairs)")
plt.xlabel("Target Win Probability (AI Perspective)")
plt.ylabel("Positions")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 3. Widened BatchNorm-Free 3-Layer Quantized Value CNN (`SVGChessValueCNN` v2)
#    `6 -> 18 (5x5 Knight-Vision) -> 27 (3x3) -> 36 (2x2) + 3 Linear Skip = 2,623 Params`
# ---------------------------------------------------------------------------

def ste_q8(x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
    """Exact SVG intermediate RGBA8888 buffer physics: clamp to [0, 1] + 8-bit uint8 STE rounding."""
    xc = torch.clamp(x, 0.0, 1.0)
    if not quantize:
        return xc
    xq = torch.round(xc * 255.0) / 255.0
    return xc + (xq - xc).detach()


class SVGPointwise1x1(nn.Module):
    """Pointwise 1x1 cross-channel mixing matching SVG `<feColorMatrix>` RGB triplets
    summed around exact integer zero-point `Z0 = 128/255` via `<feComposite k2=1 k3=1 k4=-Z0>`,
    followed by per-channel `<feComponentTransfer type='linear'>` rescaling into `[0, 1]`.
    """

    def __init__(self, in_channels: int, out_channels: int, init_gain: float = 1.0, center_input: bool = False):
        super().__init__()
        assert in_channels % 3 == 0 and out_channels % 3 == 0
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels))
        self.bias = nn.Parameter(torch.zeros(out_channels))
        nn.init.kaiming_normal_(self.weight, mode="fan_in", nonlinearity="relu")
        with torch.no_grad():
            self.weight.mul_(init_gain)
            if center_input:
                self.bias.copy_(0.06 - self.weight.sum(dim=1) * ZERO_POINT)
            else:
                self.bias.fill_(0.03)

    def compute_channel_scales(self) -> torch.Tensor:
        with torch.no_grad():
            w_pos = torch.clamp(self.weight, min=0.0).sum(dim=1)
            w_neg = torch.clamp(-self.weight, min=0.0).sum(dim=1)
            return torch.clamp(torch.maximum(w_pos / 0.44, w_neg / 0.45), min=1.0)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        g_in = self.in_channels // 3
        z0 = ZERO_POINT
        s_ch = self.compute_channel_scales()
        s_col = s_ch.unsqueeze(1)
        s_view = s_ch.view(1, self.out_channels, 1, 1)

        w0 = (self.weight[:, 0:3] / s_col).view(self.out_channels, 3, 1, 1)
        acc = ste_q8(z0 + F.conv2d(x[:, 0:3], w0), quantize=quantize)
        for g in range(1, g_in):
            wg = (self.weight[:, 3 * g : 3 * (g + 1)] / s_col).view(self.out_channels, 3, 1, 1)
            part = ste_q8(z0 + F.conv2d(x[:, 3 * g : 3 * (g + 1)], wg), quantize=quantize)
            acc = ste_q8(acc + part - z0, quantize=quantize)

        b_view = self.bias.view(1, self.out_channels, 1, 1)
        return ste_q8(s_view * acc + (b_view - z0 * s_view), quantize=quantize)


class SVGLinearSkip6to3(nn.Module):
    """Lossless centered 6 -> 3 piece/material & Queen-conservation projection around Z0 = 128/255
    aggregated across 8x8 -> 4x4 -> 2x2 -> 1x1 via centered 2x2 `<feConvolveMatrix>` sums.
    """

    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(3, 6))
        with torch.no_grad():
            # Channel 0: Classical piece values (P=1, N=3.1, B=3.3, R=5.0, Q=9.4, K=0)
            self.weight[0] = torch.tensor([0.043, 0.133, 0.142, 0.215, 0.405, 0.0])
            # Channel 1: Queen & Major-piece conservation vs Minors (explicitly penalizes QxN / RxN!)
            self.weight[1] = torch.tensor([-0.030, -0.095, -0.095, 0.180, 0.420, 0.0])
            # Channel 2: Minor/Bishop pair & Pawn structure balance
            self.weight[2] = torch.tensor([0.050, 0.145, 0.165, 0.120, 0.240, 0.0])

    def effective_weight(self) -> torch.Tensor:
        w = torch.clamp(self.weight, -0.44, 0.44)
        l1 = w.abs().sum(dim=1, keepdim=True).clamp(min=0.95) / 0.95
        return w / l1

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        z0 = ZERO_POINT
        w = self.effective_weight()
        w0 = w[:, 0:3].view(3, 3, 1, 1)
        off0 = (z0 - w[:, 0:3].sum(dim=1) * z0).view(1, 3, 1, 1)
        p0 = ste_q8(F.conv2d(x[:, 0:3], w0) + off0, quantize=quantize)

        w1 = w[:, 3:6].view(3, 3, 1, 1)
        off1 = (z0 - w[:, 3:6].sum(dim=1) * z0).view(1, 3, 1, 1)
        p1 = ste_q8(F.conv2d(x[:, 3:6], w1) + off1, quantize=quantize)

        s8 = ste_q8(p0 + p1 - z0, quantize=quantize)
        s4 = ste_q8((4.0 / 1.25) * F.avg_pool2d(s8, 2, 2) - round(2.2 * z0, 6), quantize=quantize)
        s2 = ste_q8((4.0 / 1.60) * F.avg_pool2d(s4, 2, 2) - round(1.5 * z0, 6), quantize=quantize)
        s1 = ste_q8(4.0 * F.avg_pool2d(s2, 2, 2) - round(3.0 * z0, 6), quantize=quantize)
        return s1


class SVGChessValueCNN(nn.Module):
    """Widened 3-Layer Quantized Chess Value Network (`2,623` params, `0` BatchNorm)
    with `5x5` Layer-1 Knight-vision kernels for HTML5 SVG 2-Ply Minimax.
    """

    def __init__(self, c1: int = 18, c2: int = 27, c3: int = 36):
        super().__init__()
        self.c1, self.c2, self.c3 = c1, c2, c3
        # Layer 1: 6 -> 18 (6 RGB tiles) + 5x5 depthwise (padding=2: exact Knight & King-shield vision!) + 2x2 pool
        self.mix1 = SVGPointwise1x1(6, c1, init_gain=1.1, center_input=True)
        self.dwconv1 = nn.Conv2d(c1, c1, kernel_size=5, stride=1, padding=2, groups=c1, bias=True)

        # Layer 2: 18 -> 27 (9 RGB tiles) + 3x3 depthwise (padding=1) + 2x2 pool (4x4 -> 2x2)
        self.mix2 = SVGPointwise1x1(c1, c2, init_gain=1.15)
        self.dwconv2 = nn.Conv2d(c2, c2, kernel_size=3, stride=1, padding=1, groups=c2, bias=True)

        # Layer 3: 27 -> 36 (12 RGB tiles) + 2x2 global depthwise aggregator (2x2 -> 1x1)
        self.mix3 = SVGPointwise1x1(c2, c3, init_gain=1.15)
        self.dwconv3 = nn.Conv2d(c3, c3, kernel_size=2, stride=1, padding=0, groups=c3, bias=True)

        # Lossless linear global material & Queen-conservation skip (3 channels = 1 RGB tile)
        self.skip_mat = SVGLinearSkip6to3()

        # Scalar Value Head: 36 + 3 = 39 features (13 RGB triplets) -> 1 logit
        self.fc = nn.Linear(c3 + 3, 1, bias=True)

        for m in (self.dwconv1, self.dwconv2, self.dwconv3):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
            with torch.no_grad():
                m.bias.fill_(0.02)
        with torch.no_grad():
            self.fc.weight.zero_()
            self.fc.weight[0, c3] = 26.0      # Strong classical material anchor
            self.fc.weight[0, c3 + 1] = 4.0   # Queen conservation anchor
            self.fc.bias.zero_()

    def compute_logit_half_range(self) -> torch.Tensor:
        with torch.no_grad():
            w_conv = self.fc.weight[:, : self.c3]
            w_skip_l1 = self.fc.weight[:, self.c3 :].abs().sum(dim=1) * 0.502
            w_pos = torch.clamp(w_conv, min=0.0).sum(dim=1) + w_skip_l1 + torch.clamp(self.fc.bias, min=0.0)
            w_neg = torch.clamp(-w_conv, min=0.0).sum(dim=1) + w_skip_l1 + torch.clamp(-self.fc.bias, min=0.0)
            m_req = torch.maximum(w_pos / (2.0 * 0.44), w_neg / (2.0 * 0.45)).amax()
            return torch.clamp(m_req, min=5.0)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        skip = self.skip_mat(x, quantize=quantize)                          # (B, 3, 1, 1) centered at Z0

        h = self.mix1(x, quantize=quantize)                                 # (B, 18, 8, 8)
        h = ste_q8(self.dwconv1(h), quantize=quantize)                      # (B, 18, 8, 8) [5x5 Knight-vision!]
        h = ste_q8(F.avg_pool2d(h, 2, 2), quantize=quantize)                # (B, 18, 4, 4)

        h = self.mix2(h, quantize=quantize)                                 # (B, 27, 4, 4)
        h = ste_q8(self.dwconv2(h), quantize=quantize)                      # (B, 27, 4, 4)
        h = ste_q8(F.avg_pool2d(h, 2, 2), quantize=quantize)                # (B, 27, 2, 2)

        h = self.mix3(h, quantize=quantize)                                 # (B, 36, 2, 2)
        h = ste_q8(self.dwconv3(h), quantize=quantize)                      # (B, 36, 1, 1)

        z0 = ZERO_POINT
        if not quantize:
            feat_c = torch.cat([h.flatten(1), skip.flatten(1) - z0], dim=1) # (B, 39)
            return self.fc(feat_c).squeeze(-1)

        feat = torch.cat([h, skip], dim=1).flatten(1)                       # (B, 39)
        m_half = self.compute_logit_half_range()
        two_m = 2.0 * m_half
        n_groups = feat.shape[1] // 3
        acc = ste_q8(
            z0 + (self.fc.bias.unsqueeze(0) + feat[:, 0:3] @ self.fc.weight[:, 0:3].t()) / two_m,
            quantize=True,
        )
        for g in range(1, n_groups):
            wg = self.fc.weight[:, 3 * g : 3 * (g + 1)]
            if g == n_groups - 1:
                part = ste_q8(z0 + ((feat[:, 3 * g : 3 * (g + 1)] - z0) @ wg.t()) / two_m, quantize=True)
            else:
                part = ste_q8(z0 + (feat[:, 3 * g : 3 * (g + 1)] @ wg.t()) / two_m, quantize=True)
            acc = ste_q8(acc + part - z0, quantize=True)
        return ((acc - z0) * two_m).squeeze(-1)


model = SVGChessValueCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"SVGChessValueCNN v2 initialized | Total trainable parameters: {total_params:,} (0 BatchNorm layers)")


In [ ]:
# ---------------------------------------------------------------------------
# 4. Train with 8-Bit STE QAT + 2-Ply Blunder Ranking + Counterfactual QxN Monotonicity
# ---------------------------------------------------------------------------

EPOCHS = 32
optimizer = torch.optim.AdamW(model.parameters(), lr=3.2e-3, weight_decay=1.5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=5e-5)

POS_F = POS_VAL_U8 / 255.0
NEG_F = NEG_VAL_U8 / 255.0


def counterfactual_material_monotonicity_loss(xb: torch.Tensor, logits_orig: torch.Tensor, use_qat: bool) -> torch.Tensor:
    """Ceteris-Paribus piece-drop & Queen-for-Knight trade monotonicity regularizer!
    Prevents convolutional layers from ever hallucinating fake counterplay when an AI piece is lost
    or when an AI Queen is traded for an Opponent Knight/Bishop on the exact same board.
    """
    sub_n = min(96, xb.shape[0])
    x_cf = xb[:sub_n].clone()
    req_margin = torch.zeros(sub_n, device=xb.device, dtype=torch.float32)

    with torch.no_grad():
        # Half 1: Counterfactual QxN / QxB trade (remove 1 AI Queen AND 1 Opponent Knight/Bishop!)
        half = sub_n // 2
        ai_q = (x_cf[:half, 4].view(half, 64) > 0.90)
        opp_nb = (x_cf[:half, 1].view(half, 64) < 0.10) | (x_cf[:half, 2].view(half, 64) < 0.10)
        has_both = ai_q.any(dim=1) & opp_nb.any(dim=1)
        for i in torch.where(has_both)[0]:
            q_sq = torch.where(ai_q[i])[0][0]
            x_cf[i, 4].view(64)[q_sq] = ZERO_POINT
            n_sqs = torch.where(x_cf[i, 1].view(64) < 0.10)[0]
            if len(n_sqs) > 0:
                x_cf[i, 1].view(64)[n_sqs[0]] = ZERO_POINT
            else:
                b_sqs = torch.where(x_cf[i, 2].view(64) < 0.10)[0]
                x_cf[i, 2].view(64)[b_sqs[0]] = ZERO_POINT
            req_margin[i] = 1.65  # Losing Q for N/B (-6 pts) must drop logit by >= 1.65!

        # Half 2: Pure single-piece hang (remove 1 AI Queen, Rook, Bishop, or Knight!)
        ch_margins = ((4, 2.30), (3, 1.35), (2, 0.85), (1, 0.85))
        for j in range(half, sub_n):
            ch, m_val = ch_margins[j % 4]
            pcs = torch.where(x_cf[j, ch].view(64) > 0.90)[0]
            if len(pcs) > 0:
                x_cf[j, ch].view(64)[pcs[0]] = ZERO_POINT
                req_margin[j] = m_val

    active = req_margin > 0.0
    if not active.any():
        return xb.new_tensor(0.0)
    logits_cf = model(x_cf[active], quantize=use_qat)
    return F.relu(req_margin[active] - (logits_orig[:sub_n][active] - logits_cf)).mean()


@torch.no_grad()
def evaluate_value_net(loader: DataLoader, pairs_x: np.ndarray, pairs_y: np.ndarray, quantize: bool = True):
    model.eval()
    preds_list, tgts_list = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        probs = torch.sigmoid(model(xb, quantize=quantize)).cpu().numpy()
        preds_list.append(probs)
        tgts_list.append(yb.numpy())
    preds = np.concatenate(preds_list)
    tgts = np.concatenate(tgts_list)
    mae = float(np.mean(np.abs(preds - tgts)))
    corr = float(np.corrcoef(preds, tgts)[0, 1])

    p_outs = []
    px_flat = pairs_x.reshape(-1, 6, 8, 8)
    for i in range(0, len(px_flat), 512):
        pb = torch.from_numpy(px_flat[i : i + 512]).to(device)
        p_outs.append(torch.sigmoid(model(pb, quantize=quantize)).cpu().numpy())
    p_out = np.concatenate(p_outs, axis=0).reshape(-1, 2)
    diff_true = pairs_y[:, 0] - pairs_y[:, 1]
    mask_sig = np.abs(diff_true) >= 0.08
    pair_acc = float(np.mean(np.sign(p_out[mask_sig, 0] - p_out[mask_sig, 1]) == np.sign(diff_true[mask_sig]))) if mask_sig.any() else 0.0
    return mae, corr, pair_acc


history = {"train_loss": [], "val_mae": [], "val_corr": [], "val_pair_acc": []}
best_score = -1e9
best_state = None
best_metrics = (1.0, 0.0, 0.0)

for epoch in range(1, EPOCHS + 1):
    model.train()
    use_qat = (epoch >= 3)
    run_loss, steps = 0.0, 0
    pair_iter = iter(pair_loader)

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()

        logits = model(xb, quantize=use_qat)
        probs = torch.sigmoid(logits)
        # 1. Calibrated Stockfish WDL loss (BCE + Huber)
        loss_wdl = F.binary_cross_entropy_with_logits(logits, yb) + 0.55 * F.smooth_l1_loss(probs, yb, beta=0.04)

        # 2. 2-Ply Good-vs-Blunder / Bad-Trade / Bad-Positional Contrastive Ranking Loss
        try:
            xpb, ypb = next(pair_iter)
        except StopIteration:
            pair_iter = iter(pair_loader)
            xpb, ypb = next(pair_iter)
        xpb, ypb = xpb.to(device), ypb.to(device)
        lp = model(xpb.view(-1, 6, 8, 8), quantize=use_qat).view(-1, 2)
        loss_pair_wdl = F.binary_cross_entropy_with_logits(lp, ypb)
        gap = ypb[:, 0] - ypb[:, 1]
        sign_t = torch.sign(gap)
        # Weight major blunders (|gap| >= 0.25, e.g. QxN!) extra heavily!
        weight_pair = (gap.abs() >= 0.04).float() * (1.0 + 1.2 * (gap.abs() >= 0.25).float())
        margin = torch.clamp(gap.abs() * 3.6, max=1.85)
        loss_rank = (weight_pair * F.relu(margin - sign_t * (lp[:, 0] - lp[:, 1]))).mean()

        # 3. Counterfactual QxN & Piece-Drop Monotonicity Loss
        loss_mono = counterfactual_material_monotonicity_loss(xb, logits, use_qat=use_qat)

        loss = loss_wdl + 0.55 * loss_pair_wdl + 0.45 * loss_rank + 0.25 * loss_mono
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.5)
        optimizer.step()

        run_loss += loss.item()
        steps += 1

    scheduler.step()
    val_mae, val_corr, val_pacc = evaluate_value_net(val_loader, xp_val_f, yp_val_f, quantize=True)
    history["train_loss"].append(run_loss / max(1, steps))
    history["val_mae"].append(val_mae)
    history["val_corr"].append(val_corr)
    history["val_pair_acc"].append(val_pacc)

    score = val_corr + 1.25 * val_pacc - 2.0 * val_mae
    marker = ""
    if use_qat and score >= best_score:
        best_score = score
        best_state = deepcopy(model.state_dict())
        best_metrics = (val_mae, val_corr, val_pacc)
        marker = " * [best]"

    mode_s = "8-bit QAT" if use_qat else "Warmup   "
    print(
        f"Epoch {epoch:02d}/{EPOCHS:02d} ({mode_s}) | "
        f"Loss: {history['train_loss'][-1]:.4f} | "
        f"Val WDL MAE: {val_mae:.4f} | "
        f"SF Corr r: {val_corr:.4f} | "
        f"2-Ply Pair Rank Acc: {val_pacc * 100:5.2f}%{marker}"
    )

model.load_state_dict(best_state)
print(
    f"\nLoaded best QAT checkpoint -> Val MAE: {best_metrics[0]:.4f} | "
    f"Stockfish r: {best_metrics[1]:.4f} | 2-Ply Contrastive Ranking: {best_metrics[2] * 100:.2f}%"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.4))
ep_x = range(1, EPOCHS + 1)
ax1.plot(ep_x, history["train_loss"], label="Train Distillation + Blunder Loss")
ax1.plot(ep_x, history["val_mae"], label="Val WDL MAE")
ax1.set_xlabel("Epoch")
ax1.set_title("Loss & WDL Calibration Error")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep_x, [c * 100 for c in history["val_corr"]], label="Stockfish Pearson r (%)")
ax2.plot(ep_x, [p * 100 for p in history["val_pair_acc"]], label="2-Ply Blunder/Move Rank Acc (%)")
ax2.set_xlabel("Epoch")
ax2.set_title("8-Bit QAT Correlation & Anti-Blunder Ranking")
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------------------
# 5. Compile Trained Model to SVG `<feConvolveMatrix>` & `<feColorMatrix>` Spec
#    + Bit-Exact 8-Bit NumPy SVG Simulator
# ---------------------------------------------------------------------------

def flip_kernel_2d(k2d: np.ndarray) -> List[float]:
    """Flip 2D spatial kernel 180 degrees so W3C `<feConvolveMatrix>` evaluates exact cross-correlation."""
    return [round(float(v), 6) for v in k2d[::-1, ::-1].reshape(-1)]


def export_depthwise_conv(conv: nn.Conv2d) -> dict:
    w = conv.weight.detach().cpu().numpy()
    b = conv.bias.detach().cpu().numpy()
    kh, kw = int(w.shape[2]), int(w.shape[3])
    tx = (kw - 1) // 2 if conv.padding[1] > 0 else 0
    ty = (kh - 1) // 2 if conv.padding[0] > 0 else 0
    filters = []
    for c in range(w.shape[0]):
        filters.append({
            "channel": int(c),
            "order": [kw, kh],
            "targetX": int(tx),
            "targetY": int(ty),
            "divisor": 1.0,
            "bias": round(float(b[c]), 6),
            "kernelMatrix": flip_kernel_2d(w[c, 0]),
            "padding": int(conv.padding[0]),
        })
    return {
        "num_channels": int(w.shape[0]),
        "svg_order": [kw, kh],
        "targetX": int(tx),
        "targetY": int(ty),
        "padding": int(conv.padding[0]),
        "filters": filters,
    }


def export_pointwise(pw: SVGPointwise1x1) -> dict:
    w = pw.weight.detach().cpu().numpy()
    b = pw.bias.detach().cpu().numpy()
    s_ch = pw.compute_channel_scales().cpu().numpy()
    z0 = ZERO_POINT
    g_in = pw.in_channels // 3
    g_out = pw.out_channels // 3
    triplets = []
    for o in range(g_out):
        s_vec = [float(s_ch[3 * o + r]) for r in range(3)]
        b_slice = b[3 * o : 3 * (o + 1)]
        color_matrices = []
        for g in range(g_in):
            blk = w[3 * o : 3 * (o + 1), 3 * g : 3 * (g + 1)] / np.array(s_vec, dtype=np.float64)[:, None]
            mat20 = [
                round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, z0,
                round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, z0,
                round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, z0,
                0.0, 0.0, 0.0, 0.0, 1.0,
            ]
            color_matrices.append(mat20)
        triplets.append({
            "out_triplet": int(o),
            "scales": [round(s, 6) for s in s_vec],
            "rescale_slopes": [round(s, 6) for s in s_vec],
            "rescale_intercepts": [round(float(b_slice[r] - z0 * s_vec[r]), 6) for r in range(3)],
            "color_matrices": color_matrices,
        })
    return {"in_channels": pw.in_channels, "out_channels": pw.out_channels, "zero_point": z0, "triplets": triplets}


def export_linear_skip(skip: SVGLinearSkip6to3) -> dict:
    w = skip.effective_weight().detach().cpu().numpy()
    z0 = ZERO_POINT
    mats = []
    for g in range(2):
        blk = w[:, 3 * g : 3 * (g + 1)]
        offs = [round(float(z0 - blk[r].sum() * z0), 6) for r in range(3)]
        mats.append([
            round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, offs[0],
            round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, offs[1],
            round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, offs[2],
            0.0, 0.0, 0.0, 0.0, 1.0,
        ])
    return {
        "zero_point": z0,
        "weights_3x6": [[round(float(x), 6) for x in row] for row in w],
        "color_matrices": mats,
        "sum_stages": [
            {"order": [2, 2], "divisor": 1.25, "bias": round(-2.2 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
            {"order": [2, 2], "divisor": 1.60, "bias": round(-1.5 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
            {"order": [2, 2], "divisor": 1.00, "bias": round(-3.0 * z0, 6), "kernelMatrix": [1, 1, 1, 1]},
        ],
    }


def export_value_head(model: SVGChessValueCNN) -> dict:
    w_fc = model.fc.weight.detach().cpu().numpy()[0]
    b_fc = float(model.fc.bias.detach().cpu().item())
    z0 = ZERO_POINT
    m_half = float(model.compute_logit_half_range().cpu().item())
    two_m = 2.0 * m_half
    g_in = len(w_fc) // 3

    color_matrices = []
    for g in range(g_in):
        sl = w_fc[3 * g : 3 * (g + 1)] / two_m
        if g == 0:
            off = z0 + b_fc / two_m
        elif g == g_in - 1:
            off = z0 - float(sl.sum()) * z0
        else:
            off = z0
        row = [round(float(sl[0]), 6), round(float(sl[1]), 6), round(float(sl[2]), 6), 0.0, round(float(off), 6)]
        color_matrices.append(row + row + row + [0.0, 0.0, 0.0, 0.0, 1.0])

    grid = np.arange(256, dtype=np.float64) / 255.0
    logits_grid = (grid - z0) * two_m
    sig_probs = 1.0 / (1.0 + np.exp(-np.clip(logits_grid, -20.0, 20.0)))
    v_table_u8 = np.clip(np.round(3.0 + 250.0 * sig_probs), 3.0, 253.0) / 255.0

    return {
        "in_features": int(len(w_fc)),
        "zero_point": z0,
        "u8_min": 3,
        "u8_max": 253,
        "logit_half_range_M": round(m_half, 6),
        "color_matrices": color_matrices,
        "sigmoid_table_values": [round(float(v), 6) for v in v_table_u8],
    }


def build_chess_export_spec(model: SVGChessValueCNN) -> dict:
    return {
        "format": "nojs-svg-chess-value-cnn-v1",
        "arch_version": 2,
        "zero_point": ZERO_POINT,
        "zero_point_u8": ZERO_POINT_U8,
        "pos_val_u8": POS_VAL_U8,
        "neg_val_u8": NEG_VAL_U8,
        "input_shape": [6, 8, 8],
        "channel_order": ["PAWN", "KNIGHT", "BISHOP", "ROOK", "QUEEN", "KING"],
        "layers": {
            "mix1": export_pointwise(model.mix1),
            "dwconv1": export_depthwise_conv(model.dwconv1),
            "pool1": {"order": [2, 2], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 1, 1, 1]},
            "mix2": export_pointwise(model.mix2),
            "dwconv2": export_depthwise_conv(model.dwconv2),
            "pool2": {"order": [2, 2], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 1, 1, 1]},
            "mix3": export_pointwise(model.mix3),
            "dwconv3": export_depthwise_conv(model.dwconv3),
            "skip_mat": export_linear_skip(model.skip_mat),
            "value_head": export_value_head(model),
        },
    }


def q8_np(x: np.ndarray) -> np.ndarray:
    return np.round(np.clip(x, 0.0, 1.0) * 255.0) / 255.0


def svg_lut_eval_np(x: np.ndarray, table_vals: Sequence[float]) -> np.ndarray:
    tbl = np.asarray(table_vals, dtype=np.float64)
    n = len(tbl) - 1
    pos = np.clip(x, 0.0, 1.0) * n
    idx = np.minimum(np.floor(pos).astype(np.int64), n - 1)
    frac = pos - idx
    return q8_np(tbl[idx] * (1.0 - frac) + tbl[idx + 1] * frac)


def svg_fe_convolve_batch_np(buf: np.ndarray, order: Sequence[int], tx: int, ty: int,
                             kernel_flat: Sequence[float], divisor: float, bias: float, stride: int = 1) -> np.ndarray:
    ow, oh = int(order[0]), int(order[1])
    k = np.asarray(kernel_flat, dtype=np.float64).reshape(oh, ow)
    bsz, h, w = buf.shape
    padded = np.pad(buf, ((0, 0), (ty, oh - 1 - ty), (tx, ow - 1 - tx)), mode="constant", constant_values=0.0)
    acc = np.zeros((bsz, h, w), dtype=np.float64)
    for i in range(oh):
        for j in range(ow):
            kv = k[oh - 1 - i, ow - 1 - j]
            if kv != 0.0:
                acc += kv * padded[:, i : i + h, j : j + w]
    res = q8_np(acc / float(divisor) + float(bias))
    if stride > 1:
        return res[:, 0::stride, 0::stride]
    if oh == 2 and ow == 2 and tx == 0 and ty == 0 and h == 2 and w == 2:
        return res[:, 0:1, 0:1]
    return res


def svg_pointwise_batch_np(ch_bufs: np.ndarray, pw_spec: dict) -> np.ndarray:
    bsz, _, h, w = ch_bufs.shape
    z0 = float(pw_spec["zero_point"])
    out = np.zeros((bsz, pw_spec["out_channels"], h, w), dtype=np.float64)
    for trip in pw_spec["triplets"]:
        o = trip["out_triplet"]
        acc = None
        for g, mat20 in enumerate(trip["color_matrices"]):
            m = np.asarray(mat20, dtype=np.float64).reshape(4, 5)[:3, :]
            rgb_in = ch_bufs[:, 3 * g : 3 * (g + 1)]
            part = q8_np(np.einsum("rc,bchw->brhw", m[:, :3], rgb_in) + m[:, 4].reshape(1, 3, 1, 1))
            acc = part if acc is None else q8_np(acc + part - z0)
        slopes = np.asarray(trip["rescale_slopes"], dtype=np.float64).reshape(1, 3, 1, 1)
        intercepts = np.asarray(trip["rescale_intercepts"], dtype=np.float64).reshape(1, 3, 1, 1)
        out[:, 3 * o : 3 * (o + 1)] = q8_np(acc * slopes + intercepts)
    return out


def simulate_svg_value_batch(x_float_6x8x8: np.ndarray, spec: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Bit-exact 8-bit SVG `<filter>` simulation on `(N, 6, 8, 8)` inputs."""
    ly = spec["layers"]
    x0 = q8_np(x_float_6x8x8)

    sk_spec = ly["skip_mat"]
    z0 = float(sk_spec["zero_point"])
    m0 = np.asarray(sk_spec["color_matrices"][0], dtype=np.float64).reshape(4, 5)[:3, :]
    m1 = np.asarray(sk_spec["color_matrices"][1], dtype=np.float64).reshape(4, 5)[:3, :]
    p0 = q8_np(np.einsum("rc,bchw->brhw", m0[:, :3], x0[:, 0:3]) + m0[:, 4].reshape(1, 3, 1, 1))
    p1 = q8_np(np.einsum("rc,bchw->brhw", m1[:, :3], x0[:, 3:6]) + m1[:, 4].reshape(1, 3, 1, 1))
    sk = q8_np(p0 + p1 - z0)
    for stg in sk_spec["sum_stages"]:
        stride = 2 if sk.shape[-1] > 2 else 1
        sk = np.stack([
            svg_fe_convolve_batch_np(sk[:, c], stg["order"], 0, 0, stg["kernelMatrix"], stg["divisor"], stg["bias"], stride=stride)
            for c in range(3)
        ], axis=1)

    h = svg_pointwise_batch_np(x0, ly["mix1"])
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in ly["dwconv1"]["filters"]
    ], axis=1)
    p1_s = ly["pool1"]
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, c], p1_s["order"], 0, 0, p1_s["kernelMatrix"], p1_s["divisor"], 0.0, stride=2)
        for c in range(h.shape[1])
    ], axis=1)

    h = svg_pointwise_batch_np(h, ly["mix2"])
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in ly["dwconv2"]["filters"]
    ], axis=1)
    p2_s = ly["pool2"]
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, c], p2_s["order"], 0, 0, p2_s["kernelMatrix"], p2_s["divisor"], 0.0, stride=2)
        for c in range(h.shape[1])
    ], axis=1)

    h = svg_pointwise_batch_np(h, ly["mix3"])
    h = np.stack([
        svg_fe_convolve_batch_np(h[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in ly["dwconv3"]["filters"]
    ], axis=1)

    feat = np.concatenate([h[:, :, 0, 0], sk[:, :, 0, 0]], axis=1)
    vh = ly["value_head"]
    z0_vh = float(vh["zero_point"])
    acc = None
    for g, mat20 in enumerate(vh["color_matrices"]):
        m = np.asarray(mat20, dtype=np.float64).reshape(4, 5)[0, :]
        part = q8_np(feat[:, 3 * g : 3 * (g + 1)] @ m[:3] + m[4])
        acc = part if acc is None else q8_np(acc + part - z0_vh)

    two_m = 2.0 * float(vh["logit_half_range_M"])
    decoded_logits = (acc - z0_vh) * two_m
    v_svg = svg_lut_eval_np(acc, vh["sigmoid_table_values"])
    v_u8 = np.round(v_svg * 255.0).astype(np.int32)
    probs = (v_svg * 255.0 - 3.0) / 250.0
    return decoded_logits, probs, v_u8


export_spec = build_chess_export_spec(model)
print("Compiled SVG `<filter>` Chess Value CNN v2 export spec successfully!")
print(f"  Channels: 6 -> {model.c1} (5x5) -> {model.c2} (3x3) -> {model.c3} (2x2) + 3 skip = {model.c3 + 3} head features")
print(f"  Logit half-range M = +-{export_spec['layers']['value_head']['logit_half_range_M']}")
print(f"  Linear skip row 0 (material) = {export_spec['layers']['skip_mat']['weights_3x6'][0]}")
print(f"  Linear skip row 1 (Q-conserv)= {export_spec['layers']['skip_mat']['weights_3x6'][1]}")


In [ ]:
# ---------------------------------------------------------------------------
# 6. Bit-Exact 8-Bit SVG Parity Check & 2-Ply Anti-Blunder / QxN Tactical Suite
# ---------------------------------------------------------------------------

@torch.no_grad()
def get_pytorch_qat_logits(x_np: np.ndarray) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(x_np), 512):
        xb = torch.from_numpy(x_np[i : i + 512]).to(device)
        outs.append(model(xb, quantize=True).cpu().numpy())
    return np.concatenate(outs, axis=0)


qat_logits_test = get_pytorch_qat_logits(x_test_f)
qat_probs_test = 1.0 / (1.0 + np.exp(-np.clip(qat_logits_test, -20.0, 20.0)))

svg_logits_chunks, svg_probs_chunks, svg_u8_chunks = [], [], []
for i in range(0, len(x_test_f), 1024):
    lg_c, pr_c, u8_c = simulate_svg_value_batch(x_test_f[i : i + 1024], export_spec)
    svg_logits_chunks.append(lg_c)
    svg_probs_chunks.append(pr_c)
    svg_u8_chunks.append(u8_c)
svg_logits_test = np.concatenate(svg_logits_chunks)
svg_probs_test = np.concatenate(svg_probs_chunks)
svg_u8_test = np.concatenate(svg_u8_chunks)

logit_abs_diff = np.abs(qat_logits_test - svg_logits_test)
lsb_step = (2.0 * float(export_spec["layers"]["value_head"]["logit_half_range_M"])) / 255.0
lsb_diff = np.round(logit_abs_diff / max(lsb_step, 1e-9))

test_mae_qat = float(np.mean(np.abs(qat_probs_test - y_test_f)))
test_corr_qat = float(np.corrcoef(qat_probs_test, y_test_f)[0, 1])
test_mae_svg = float(np.mean(np.abs(svg_probs_test - y_test_f)))
test_corr_svg = float(np.corrcoef(svg_probs_test, y_test_f)[0, 1])
qat_svg_mae_gap = abs(test_mae_qat - test_mae_svg)
qat_svg_corr_gap = abs(test_corr_qat - test_corr_svg)

LVA_ORDER = {chess.PAWN: 1, chess.KNIGHT: 2, chess.BISHOP: 3, chess.ROOK: 4, chess.QUEEN: 5, chess.KING: 6}
MVV_TIER = {chess.PAWN: 1, chess.KNIGHT: 2, chess.BISHOP: 2, chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 4}


def select_move_2ply_svg_minimax(board: chess.Board, spec: dict) -> Tuple[chess.Move, int, Dict[str, int]]:
    """Exact 2-Ply Minimax matching the HTML5 SVG 16-quadrant MVV-LVA selector (`s1a..s5b`)."""
    ai_col = board.turn
    m1_by_dst: Dict[int, chess.Move] = {}
    for mv in board.legal_moves:
        if mv.promotion not in (None, chess.QUEEN):
            continue
        cur = m1_by_dst.get(mv.to_square)
        if cur is None or LVA_ORDER[board.piece_type_at(mv.from_square)] < LVA_ORDER[board.piece_type_at(cur.from_square)]:
            m1_by_dst[mv.to_square] = mv

    q_v1: Dict[int, int] = {v1: 255 for v1 in m1_by_dst}
    leaf_boards_u8: List[np.ndarray] = []
    leaf_map: List[int] = []

    for v1, m1 in m1_by_dst.items():
        b1 = board.copy(stack=False)
        b1.push(m1)
        if b1.is_checkmate():
            q_v1[v1] = 254
            continue
        if b1.is_stalemate() or b1.is_insufficient_material():
            q_v1[v1] = ZERO_POINT_U8
            continue

        # Exact 16-quadrant MVV-LVA reply selector (matches `s1h..s1i` on the HTML5 SVG stage!)
        quad_best: Dict[Tuple[int, int], Tuple[int, chess.Move]] = {}
        for m2 in b1.legal_moves:
            if m2.promotion not in (None, chess.QUEEN):
                continue
            r2, f2 = divmod(m2.to_square, 8)
            ar2 = r2 if ai_col == chess.WHITE else (7 - r2)
            vic = b1.piece_type_at(m2.to_square)
            mvv = MVV_TIER[vic] if vic is not None else 0
            pt_s = LVA_ORDER[b1.piece_type_at(m2.from_square)]
            if mvv > 0:
                bt = (31 if mvv == 4 else (25 + mvv)) if (pt_s == 1 and ar2 == 0) else ((25 if mvv == 4 else 6 * mvv) + (6 - pt_s))
            else:
                bt = 24 if (pt_s == 1 and ar2 == 0) else {6: 0, 1: 1, 5: 2, 4: 3, 3: 4, 2: 5}[pt_s]
            sc2 = 128 + 4 * bt + 2 * (ar2 % 2) + (f2 % 2)
            qk = (ar2 // 2, f2 // 2)
            if qk not in quad_best or sc2 > quad_best[qk][0]:
                quad_best[qk] = (sc2, m2)

        for _sc2, m2 in quad_best.values():
            b2 = b1.copy(stack=False)
            b2.push(m2)
            leaf_boards_u8.append(encode_board_uint8(b2, ai_color=ai_col))
            leaf_map.append(v1)

    if leaf_boards_u8:
        x_leaves = np.stack(leaf_boards_u8, axis=0).astype(np.float32) / 255.0
        _, _, u8_scores = simulate_svg_value_batch(x_leaves, spec)
        for idx, v1 in enumerate(leaf_map):
            val = int(u8_scores[idx])
            if val < q_v1[v1]:
                q_v1[v1] = val

    best_v1 = max(q_v1.keys(), key=lambda v: (q_v1[v], -abs((v % 8) - 3.5) - abs((v // 8) - 3.5)))
    uci_scores = {m1_by_dst[v].uci(): q_v1[v] for v in q_v1}
    return m1_by_dst[best_v1], q_v1[best_v1], uci_scores


print("=====================================================================")
print(f"  Held-Out Test Set Size:                 {len(y_test_f):,} unseen Stockfish positions")
print(f"  PyTorch 8-Bit QAT WDL MAE / Corr r:     {test_mae_qat:.4f} / {test_corr_qat:.4f}")
print(f"  Exact 8-Bit SVG WDL MAE / Corr r:       {test_mae_svg:.4f} / {test_corr_svg:.4f}  (gap: {qat_svg_mae_gap:.4f} / {qat_svg_corr_gap:.4f})")
print(f"  FP32-QAT vs FP64-SVG Byte Diff (LSBs):  Median {np.median(lsb_diff):.0f} LSB | 99th pct {np.percentile(lsb_diff, 99):.0f} LSB | Max {lsb_diff.max():.0f} LSB")
print("=====================================================================")
assert test_corr_svg >= 0.88, f"Expected >= 0.88 Stockfish Pearson r, got {test_corr_svg:.4f}"
assert qat_svg_mae_gap < 0.008 and qat_svg_corr_gap < 0.008
assert np.median(lsb_diff) <= 2

TACTICAL_AND_ANTI_BLUNDER_SUITE = [
    ("Capture Hanging Queen", "r1bqkbnr/ppp2ppp/2np4/4p3/4P1Q1/2N2N2/PPPP1PPP/R1B1KB1R b KQkq - 1 4", "exact", {"c8g4"}),
    ("Refuse QxN Trade #1 (Qd4 vs defended Nd5)", "rnbqkbnr/pppp1ppp/4p3/3n4/3Q4/2N5/PPPPPPPP/R1B1KBNR w KQkq - 0 3", "forbid", {"d4d5"}),
    ("Refuse QxN Trade #2 (Qa5 vs defended Nc3)", "rnb1kbnr/ppp2ppp/3p4/q3p3/4P3/2N2N2/PPPP1PPP/R1BQKB1R b KQkq - 2 4", "forbid", {"a5c3"}),
    ("Save Attacked Queen (Do NOT sacrifice QxN!)", "r1bqkb1r/pppp1ppp/2n2n2/3Np3/3qP3/5N2/PPPP1PPP/R1BQKB1R b KQkq - 3 4", "save_q", {chess.D4}),
    ("Refuse Defended-Pawn Bait & Save Queen", "rnb1kbnr/pppp1ppp/8/4p3/4P2q/2N2N2/PPPP1PPP/R1BQKB1R b KQkq - 3 3", "save_q", {chess.H4}),
    ("Deliver Mate in 1 (Back-Rank Mate)", "6k1/5ppp/8/8/8/8/5PPP/3R2K1 w - - 0 1", "exact", {"d1d8"}),
    ("Win Undefended Rook", "4k3/p7/8/3b4/8/8/8/4K2R b - - 0 1", "exact", {"d5h1"}),
    ("Avoid Stalemate Trap & Deliver Mate", "7k/5K2/8/6Q1/8/8/8/8 w - - 0 1", "exact", {"g5h4", "g5h5", "g5h6", "g5g7", "g5f6"}),
]

print("\n2-Ply SVG CNN Minimax Tactical & Anti-QxN Blunder Verification:")
for name, fen, mode, ref_set in TACTICAL_AND_ANTI_BLUNDER_SUITE:
    bd = chess.Board(fen)
    mv, sc, uci_sc = select_move_2ply_svg_minimax(bd, export_spec)
    if mode == "exact":
        ok = mv.uci() in ref_set
        detail = f"played {mv.uci()} (Q={sc}/255)"
    elif mode == "forbid":
        bad_u = next(iter(ref_set))
        ok = mv.uci() not in ref_set and sc > uci_sc.get(bad_u, 0) + 25
        detail = f"played {mv.uci()} (Q={sc}) vs forbidden {bad_u} (Q={uci_sc.get(bad_u)})"
    else:
        q_sq = next(iter(ref_set))
        b_af = bd.copy(); b_af.push(mv)
        ok = (mv.from_square == q_sq) and not b_af.is_attacked_by(not bd.turn, mv.to_square)
        detail = f"played {mv.uci()} (Q={sc}/255, Queen safe={ok})"
    print(f"  [{'PASS' if ok else 'WARN'}] {name:46s} -> {detail}")


In [ ]:
# ---------------------------------------------------------------------------
# 7. Export `weights.json` (with 24 Golden SVG Test Vectors) & Trigger Download
# ---------------------------------------------------------------------------

golden_positions = []
for idx in range(24):
    fen_i = SINGLE_FENS[te_idx[idx]]
    golden_positions.append({
        "id": f"chess_test_{idx:02d}",
        "fen": fen_i,
        "regime": regimes_test[idx],
        "stockfish_wdl": round(float(y_test_f[idx]), 5),
        "svg_u8": int(svg_u8_test[idx]),
        "svg_prob": round(float(svg_probs_test[idx]), 5),
        "svg_logit": round(float(svg_logits_test[idx]), 4),
    })

export_spec["metrics"] = {
    "val_wdl_mae_qat": round(float(best_metrics[0]), 5),
    "val_stockfish_corr_r": round(float(best_metrics[1]), 5),
    "val_sibling_rank_acc": round(float(best_metrics[2]), 5),
    "test_wdl_mae_svg_uint8": round(float(test_mae_svg), 5),
    "test_stockfish_corr_svg_uint8": round(float(test_corr_svg), 5),
    "qat_to_svg_median_lsb_diff": int(round(float(np.median(lsb_diff)))),
    "qat_to_svg_max_lsb_diff": int(round(float(lsb_diff.max()))),
    "qat_to_svg_max_logit_diff": round(float(logit_abs_diff.max()), 6),
    "total_parameters": int(total_params),
    "num_labeled_positions": int(len(combined_fens)),
}
export_spec["golden_positions"] = golden_positions

OUT_PATH = "weights.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(export_spec, f, separators=(",", ":"))

size_kb = os.path.getsize(OUT_PATH) / 1024.0
print(
    f"Saved {OUT_PATH} ({size_kb:.1f} KB, {total_params:,} params) | "
    f"Test Stockfish r = {test_corr_svg:.4f} | Test MAE = {test_mae_svg:.4f}"
)

try:
    from google.colab import files
    files.download(OUT_PATH)
    print("Triggered browser download for weights.json!")
except ImportError:
    print("Not running inside Google Colab; weights.json written to current directory.")
